# Sketch2Shape AI — Hunyuan3D-2 GPU backend (Google Colab)

TechZephyr 4.0 · IIT Bhubaneswar · *From a 2D Sketch to a 3D Asset*

This notebook turns Colab into the GPU backend for the Sketch2Shape web app:

`sketch → SD1.5 + ControlNet Scribble (concept render) → background removal → Hunyuan3D-DiT (shape) → Hunyuan3D-Paint (texture) → trimesh validation → .glb / .obj`

**Before you start:** `Runtime → Change runtime type → GPU`.

| GPU | VRAM | Recommended shape model | Texture |
|---|---|---|---|
| T4 (free) | 15 GB | `Hunyuan3D-2mini` turbo (default) | yes, low-VRAM offload (slow) |
| L4 | 22 GB | `Hunyuan3D-2` turbo | yes |
| A100 | 40 GB | `Hunyuan3D-2` (full, 30 steps) | yes |

Run the cells top to bottom. The first run takes roughly 15–25 minutes (installs, CUDA compilation, about 15 GB of model downloads).

## 0 · Check the GPU
Runs in a subprocess on purpose: importing torch/numpy in this notebook before the installs leaves old modules in memory and causes mixed-version errors later.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!python -c "import sys, torch; print('python', sys.version.split()[0], '| torch', torch.__version__, '| cuda', torch.version.cuda, '| gpu', torch.cuda.is_available())"

## 1 · Clone and install Hunyuan3D-2
Colab already ships PyTorch with CUDA, so we keep its build and only install the repo requirements.

In [ ]:
%cd /content
!git clone --depth 1 https://github.com/Tencent-Hunyuan/Hunyuan3D-2.git
%cd /content/Hunyuan3D-2
!pip install -q -r requirements.txt
!pip install -q -e .

**Expected warning:** pip may print `dependency conflicts` for preinstalled Colab packages (`pytensor`, `cudf`, `cuml`, `jax`…). This project does not use them, so it is safe to ignore.

**Do not pin `numpy<2`, `numba<0.65` or Pillow.** On Colab's Python 3.13, rembg needs numpy>=2.3 and OpenCV 5 is built against numpy 2.

## 2 · Extra dependencies (ControlNet, API server, mesh tools)
`transformers` is pinned below 5: v5 renamed the DINOv2 layers that the Hunyuan3D-2 checkpoints expect.

In [ ]:
!pip install -q "diffusers>=0.30.0" "transformers>=4.46,<5" "accelerate>=0.33.0" \
    fastapi uvicorn python-multipart "trimesh>=4.0" rembg opencv-python-headless \
    pygltflib xatlas pymeshlab ninja pybind11

## 3 · Compile the texture rasterizer (CUDA, ~5–10 min)
Required for Hunyuan3D-Paint. If this fails, shape generation still works and the app exports untextured meshes.

In [ ]:
%cd /content/Hunyuan3D-2/hy3dgen/texgen/custom_rasterizer
!pip install -q --no-build-isolation . || python3 setup.py install
%cd /content/Hunyuan3D-2/hy3dgen/texgen/differentiable_renderer
!pip install -q --no-build-isolation . || python3 setup.py install
%cd /content/Hunyuan3D-2
# Newer diffusers require trust_remote_code=True for Hunyuan's local paint pipeline
!sed -i 's/custom_pipeline=custom_pipeline_path, torch_dtype=torch.float16)/custom_pipeline=custom_pipeline_path, torch_dtype=torch.float16, trust_remote_code=True)/' hy3dgen/texgen/utils/multiview_utils.py
!grep -c "trust_remote_code=True" hy3dgen/texgen/utils/multiview_utils.py

## 3b · Restart the session (required)
The installs above upgraded numpy, Pillow and others on disk. This cell restarts the Python process so every later cell loads one consistent set of versions.
Colab will show *"Your session crashed"*. **That is expected.** Continue with step 3c. Installed packages are kept.

In [ ]:
import os
os.kill(os.getpid(), 9)

## 3c · Verify the environment (after the restart)

In [ ]:
import numpy, numba, scipy.linalg, PIL, cv2, torch, diffusers, transformers
assert int(transformers.__version__.split(".")[0]) < 5, 'transformers>=5 breaks Hunyuan3D-2: pip install "transformers>=4.46,<5" and restart'
from diffusers import StableDiffusionControlNetPipeline  # noqa: F401
from hy3dgen.shapegen import Hunyuan3DDiTFlowMatchingPipeline  # noqa: F401
from hy3dgen.rembg import BackgroundRemover  # noqa: F401
print("OK | numpy", numpy.__version__, "| numba", numba.__version__, "| pillow", PIL.__version__,
      "| cv2", cv2.__version__, "| torch", torch.__version__, "| diffusers", diffusers.__version__, "| transformers", transformers.__version__)
try:
    import custom_rasterizer  # noqa: F401
    print("custom_rasterizer OK - texture painting enabled")
except Exception as e:
    print("custom_rasterizer NOT available - shapes will be untextured:", e)

## 4 · Write the Sketch2Shape API server
Same file as `backend/server.py` in the project repo.

In [ ]:
%%writefile /content/s2s_server.py
"""Sketch2Shape AI - Google Colab GPU backend.

Pipeline (one job at a time on the GPU):
  preprocess -> concept (SD1.5 + ControlNet Scribble) -> segment (rembg)
  -> shape (Hunyuan3D-DiT) -> texture (Hunyuan3D-Paint) -> export (trimesh)

Run inside Colab after installing Hunyuan3D-2 (see colab/Sketch2Shape_Hunyuan3D.ipynb):
    python server.py            # or import and call start_in_background()

Environment overrides:
    S2S_SHAPE_MODEL      default tencent/Hunyuan3D-2mini
    S2S_SHAPE_SUBFOLDER  default hunyuan3d-dit-v2-mini-turbo
    S2S_TEX_MODEL        default tencent/Hunyuan3D-2
    S2S_SD_MODEL         default Lykon/dreamshaper-8
    S2S_CN_MODEL         default lllyasviel/control_v11p_sd15_scribble
    S2S_ENABLE_TEXTURE   1/0 (default 1)
    S2S_SEQUENTIAL       1/0 force load-run-unload per stage (default: auto when system RAM < 20 GB)
    S2S_OUTPUT_DIR       default /content/s2s_outputs
    S2S_PORT             default 8000
"""

from __future__ import annotations

import gc
import json
import os
import queue
import threading
import time
import traceback
import uuid
import zipfile
from pathlib import Path

import numpy as np
import torch
import trimesh
from fastapi import FastAPI, File, Form, HTTPException, UploadFile
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import FileResponse
from PIL import Image, ImageFilter, ImageOps

VERSION = "1.0.0"
SHAPE_MODEL = os.environ.get("S2S_SHAPE_MODEL", "tencent/Hunyuan3D-2mini")
SHAPE_SUBFOLDER = os.environ.get("S2S_SHAPE_SUBFOLDER", "hunyuan3d-dit-v2-mini-turbo")
TEX_MODEL = os.environ.get("S2S_TEX_MODEL", "tencent/Hunyuan3D-2")
SD_MODEL = os.environ.get("S2S_SD_MODEL", "Lykon/dreamshaper-8")
CN_MODEL = os.environ.get("S2S_CN_MODEL", "lllyasviel/control_v11p_sd15_scribble")
ENABLE_TEXTURE = os.environ.get("S2S_ENABLE_TEXTURE", "1") == "1"
OUT_DIR = Path(os.environ.get("S2S_OUTPUT_DIR", "/content/s2s_outputs"))
PORT = int(os.environ.get("S2S_PORT", "8000"))

OUT_DIR.mkdir(parents=True, exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1024**3 if DEVICE == "cuda" else None
GPU_NAME = torch.cuda.get_device_name(0) if DEVICE == "cuda" else None
# Below ~20 GB (T4 / L4-16) keep only the active model on the GPU.
LOW_VRAM = VRAM_GB is None or VRAM_GB < 20


def _ram_gb() -> float | None:
    try:
        import psutil

        return psutil.virtual_memory().total / 1024**3
    except Exception:
        return None


def _ram_used_gb() -> float | None:
    try:
        import psutil

        return psutil.virtual_memory().used / 1024**3
    except Exception:
        return None


RAM_GB = _ram_gb()
# Free Colab has ~12.7 GB system RAM: CPU offload would park every model in RAM
# and crash the runtime. Sequential mode loads each model straight onto the GPU,
# runs its stage and frees it before the next stage.
_seq_env = os.environ.get("S2S_SEQUENTIAL")
SEQUENTIAL = (_seq_env == "1") if _seq_env in ("0", "1") else (RAM_GB is None or RAM_GB < 20)

NEGATIVE = (
    "multiple objects, cropped, cut off, text, watermark, busy background, scene, floor, "
    "harsh shadow, low quality, blurry, deformed, extra parts"
)
STAGES = ["preprocess", "concept", "segment", "shape", "texture", "export"]
ARTIFACT_FILES = {
    "control": "control.png",
    "concept": "concept.png",
    "segmented": "segmented.png",
    "glb": "model.glb",
    "obj_zip": "model_obj.zip",
}


# --------------------------------------------------------------------------- models
class Models:
    """Lazily loaded model registry with VRAM-aware placement."""

    def __init__(self) -> None:
        self.sd = None
        self.rembg = None
        self.shape = None
        self.paint = None
        self.lock = threading.Lock()

    def flush(self) -> None:
        gc.collect()
        if DEVICE == "cuda":
            torch.cuda.empty_cache()

    def get_sd(self):
        if self.sd is None:
            from diffusers import ControlNetModel, StableDiffusionControlNetPipeline, UniPCMultistepScheduler

            dtype = torch.float16 if DEVICE == "cuda" else torch.float32
            cn = ControlNetModel.from_pretrained(CN_MODEL, torch_dtype=dtype)
            pipe = StableDiffusionControlNetPipeline.from_pretrained(
                SD_MODEL, controlnet=cn, torch_dtype=dtype, safety_checker=None, requires_safety_checker=False
            )
            pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config)
            if DEVICE == "cuda" and LOW_VRAM and not SEQUENTIAL:
                pipe.enable_model_cpu_offload()
            else:
                pipe.to(DEVICE)
            self.sd = pipe
        return self.sd

    def get_rembg(self):
        if self.rembg is None:
            from hy3dgen.rembg import BackgroundRemover

            self.rembg = BackgroundRemover()
        return self.rembg

    def get_shape(self):
        if self.shape is None:
            from hy3dgen.shapegen import Hunyuan3DDiTFlowMatchingPipeline

            check_transformers()
            try:
                if DEVICE == "cuda" and SEQUENTIAL:
                    pipe = load_shape_low_ram()
                else:
                    pipe = Hunyuan3DDiTFlowMatchingPipeline.from_pretrained(
                        SHAPE_MODEL, subfolder=SHAPE_SUBFOLDER, use_safetensors=True, device=DEVICE
                    )
            except FileNotFoundError as exc:
                # Only fall back when the .safetensors file is missing. A key
                # mismatch would fail identically with .ckpt, which is also
                # loaded fully into RAM.
                print("[s2s] safetensors weights missing, falling back to ckpt:", exc)
                pipe = Hunyuan3DDiTFlowMatchingPipeline.from_pretrained(
                    SHAPE_MODEL, subfolder=SHAPE_SUBFOLDER, use_safetensors=False, device=DEVICE
                )
            except RuntimeError as exc:
                if "Missing key(s)" in str(exc):
                    raise RuntimeError(
                        "Hunyuan3D image-encoder weights do not match the installed transformers "
                        'version. Run: pip install "transformers>=4.46,<5", then restart the session.'
                    ) from None
                raise
            try:
                pipe.enable_flashvdm()  # faster volume decoding (turbo models)
            except Exception as exc:  # older checkouts
                print("[s2s] FlashVDM unavailable:", exc)
            self.shape = pipe
        return self.shape

    def get_paint(self):
        if not ENABLE_TEXTURE:
            return None
        if self.paint is None:
            patch_hunyuan_paint()
            from hy3dgen.texgen import Hunyuan3DPaintPipeline

            try:
                pipe = Hunyuan3DPaintPipeline.from_pretrained(TEX_MODEL, subfolder="hunyuan3d-paint-v2-0-turbo")
            except TypeError:
                pipe = Hunyuan3DPaintPipeline.from_pretrained(TEX_MODEL)
            if LOW_VRAM and not SEQUENTIAL and hasattr(pipe, "enable_model_cpu_offload"):
                pipe.enable_model_cpu_offload()
            self.paint = pipe
        return self.paint

    def release(self, name: str) -> None:
        """Drop a model completely (frees both GPU memory and system RAM)."""
        if getattr(self, name, None) is not None:
            setattr(self, name, None)
            self.flush()
            print(f"[s2s] released {name} | RAM used {_ram_used_gb() or 0:.1f} GB")

    def release_if_sequential(self, name: str) -> None:
        if SEQUENTIAL:
            self.release(name)

    def loaded(self) -> dict:
        return {
            "controlnet": self.sd is not None,
            "rembg": self.rembg is not None,
            "shape": self.shape is not None,
            "texture": self.paint is not None,
        }


MODELS = Models()


def load_shape_low_ram():
    """RAM-frugal replacement for Hunyuan3DDiTPipeline.from_single_file.

    The stock loader reads the whole checkpoint into CPU RAM and builds every
    sub-model in fp32 on the CPU (~14 GB peak for 2mini). Here each tensor is
    streamed from disk straight to the GPU, and each sub-model is built on the
    GPU and cast to fp16 before the next one, so system RAM stays near baseline.
    """
    import yaml
    from safetensors import safe_open
    from hy3dgen.shapegen import Hunyuan3DDiTFlowMatchingPipeline
    from hy3dgen.shapegen.pipelines import instantiate_from_config
    from hy3dgen.shapegen.utils import smart_load_model

    dtype = torch.float16
    config_path, ckpt_path = smart_load_model(
        SHAPE_MODEL, subfolder=SHAPE_SUBFOLDER, use_safetensors=True, variant="fp16"
    )
    ckpt_path = ckpt_path.replace(".ckpt", ".safetensors")
    if not os.path.exists(ckpt_path):
        raise FileNotFoundError(f"Model file {ckpt_path} not found")
    with open(config_path) as f:
        config = yaml.safe_load(f)

    gpu = torch.device("cuda")
    parts = {}
    with safe_open(ckpt_path, framework="pt", device="cuda") as sf:
        keys = list(sf.keys())
        for name, strict in (("model", True), ("vae", False), ("conditioner", True)):
            t0 = time.time()
            with gpu:  # parameters are allocated on the GPU, not in system RAM
                module = instantiate_from_config(config[name])
            prefix = name + "."
            state = {k[len(prefix):]: sf.get_tensor(k) for k in keys if k.startswith(prefix)}
            if state:
                module.load_state_dict(state, strict=strict)
            del state
            module.to(device=gpu, dtype=dtype)
            parts[name] = module
            gc.collect()
            torch.cuda.empty_cache()
            print(
                f"[s2s] shape/{name} on GPU in {time.time() - t0:.0f}s | "
                f"VRAM {torch.cuda.memory_allocated() / 1024**3:.1f} GB | RAM used {_ram_used_gb() or 0:.1f} GB"
            )

    return Hunyuan3DDiTFlowMatchingPipeline(
        vae=parts["vae"],
        model=parts["model"],
        conditioner=parts["conditioner"],
        scheduler=instantiate_from_config(config["scheduler"]),
        image_processor=instantiate_from_config(config["image_processor"]),
        device="cuda",
        dtype=dtype,
        # enable_flashvdm() reads these to locate the turbo VAE
        from_pretrained_kwargs=dict(
            model_path=SHAPE_MODEL,
            subfolder=SHAPE_SUBFOLDER,
            use_safetensors=True,
            variant="fp16",
            dtype=dtype,
            device="cuda",
        ),
    )


def check_transformers() -> None:
    """Hunyuan3D-2 checkpoints use transformers 4.x DINOv2 module names."""
    import transformers

    major = int(transformers.__version__.split(".")[0])
    if major >= 5:
        raise RuntimeError(
            f"transformers {transformers.__version__} renamed DINOv2 layers that Hunyuan3D-2 expects. "
            'Run: pip install "transformers>=4.46,<5", then restart the session.'
        )


def patch_hunyuan_paint() -> None:
    """Newer diffusers require trust_remote_code=True for Hunyuan's local custom
    paint pipeline. Patch the (editable-installed) source once, idempotently."""
    import importlib.util

    spec = importlib.util.find_spec("hy3dgen")
    if not spec or not spec.submodule_search_locations:
        return
    path = Path(list(spec.submodule_search_locations)[0]) / "texgen" / "utils" / "multiview_utils.py"
    if not path.exists():
        return
    src = path.read_text()
    old = "custom_pipeline=custom_pipeline_path, torch_dtype=torch.float16)"
    if "trust_remote_code" not in src and old in src:
        path.write_text(src.replace(old, old[:-1] + ", trust_remote_code=True)"))
        print("[s2s] patched", path, "with trust_remote_code=True")


patch_hunyuan_paint()


def place(pipe, device: str) -> None:
    """Move a Hunyuan pipeline between CPU/GPU when running in low-VRAM mode."""
    if SEQUENTIAL or not (LOW_VRAM and DEVICE == "cuda"):
        return
    try:
        pipe.to(device)
    except Exception as exc:
        print(f"[s2s] could not move pipeline to {device}:", exc)


def texture_available() -> bool:
    if not ENABLE_TEXTURE:
        return False
    try:
        import custom_rasterizer  # noqa: F401  compiled in the notebook

        return True
    except Exception:
        return False


# --------------------------------------------------------------------------- pipeline
def make_control(sketch: Image.Image) -> Image.Image:
    """ControlNet Scribble expects white strokes on black."""
    gray = ImageOps.grayscale(sketch).resize((512, 512), Image.LANCZOS)
    inv = ImageOps.invert(gray).point(lambda v: 255 if v > 60 else 0)
    return inv.filter(ImageFilter.MaxFilter(3)).convert("RGB")


def build_prompt(user_prompt: str) -> str:
    subject = user_prompt.strip() or "a single object"
    return (
        f"{subject}, a single complete object, centered, isolated on a plain white background, "
        "product photography, soft studio lighting, three-quarter view, high detail, sharp focus"
    )


def mesh_stats(mesh: trimesh.Trimesh) -> dict:
    return {
        "vertices": int(len(mesh.vertices)),
        "faces": int(len(mesh.faces)),
        "watertight": bool(mesh.is_watertight),
        "extents": [float(x) for x in mesh.extents],
        "textured": getattr(mesh.visual, "kind", None) == "texture",
    }


def export_obj_zip(mesh: trimesh.Trimesh, job_dir: Path) -> Path:
    zip_path = job_dir / ARTIFACT_FILES["obj_zip"]
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        try:
            text, files = trimesh.exchange.obj.export_obj(
                mesh, include_texture=True, return_texture=True, mtl_name="model.mtl"
            )
            zf.writestr("model.obj", text)
            for name, data in files.items():
                zf.writestr(name, data)
        except Exception:
            zf.writestr("model.obj", trimesh.exchange.obj.export_obj(mesh, include_texture=False))
    return zip_path


def run_job(job: dict, sketch: Image.Image, prompt: str, opts: dict) -> None:
    job_dir = OUT_DIR / job["id"]
    t_start = time.time()

    def stage(name: str, message: str):
        job["stage"], job["message"] = name, message
        return time.time()

    def done(name: str, t0: float):
        job["timings"][name] = round(time.time() - t0, 2)

    # 1. preprocess
    t = stage("preprocess", "Normalising sketch")
    sketch = ImageOps.exif_transpose(sketch).convert("RGB")
    sketch.save(job_dir / "input.png")
    control = make_control(sketch)
    control.save(job_dir / ARTIFACT_FILES["control"])
    job["artifacts"]["control"] = f"/jobs/{job['id']}/files/control"
    done("preprocess", t)

    # 2. concept render (sketch-guided image generation)
    if opts.get("use_controlnet", True):
        t = stage("concept", "Rendering concept with ControlNet")
        sd = MODELS.get_sd()
        gen = torch.Generator(device="cpu").manual_seed(int(opts.get("seed", 1234)))
        concept = sd(
            build_prompt(prompt),
            image=control,
            negative_prompt=NEGATIVE,
            num_inference_steps=int(opts.get("concept_steps", 25)),
            guidance_scale=7.5,
            controlnet_conditioning_scale=float(opts.get("controlnet_scale", 0.9)),
            generator=gen,
        ).images[0]
        concept.save(job_dir / ARTIFACT_FILES["concept"])
        job["artifacts"]["concept"] = f"/jobs/{job['id']}/files/concept"
        done("concept", t)
        del sd
        MODELS.release_if_sequential("sd")
        MODELS.flush()
    else:
        job["stages_skipped"].append("concept")
        concept = sketch

    # 3. foreground isolation
    t = stage("segment", "Removing background")
    segmented = MODELS.get_rembg()(concept.convert("RGB"))
    segmented.save(job_dir / ARTIFACT_FILES["segmented"])
    job["artifacts"]["segmented"] = f"/jobs/{job['id']}/files/segmented"
    done("segment", t)

    # 4. shape generation
    t = stage("shape", "Generating geometry with Hunyuan3D-DiT")
    shape = MODELS.get_shape()
    place(shape, "cuda")
    steps = 5 if opts.get("turbo", True) else 30
    mesh = shape(
        image=segmented,
        num_inference_steps=steps,
        octree_resolution=int(opts.get("octree_resolution", 256)),
        num_chunks=20000,
        generator=torch.manual_seed(int(opts.get("seed", 1234))),
        output_type="trimesh",
    )[0]
    from hy3dgen.shapegen import DegenerateFaceRemover, FaceReducer, FloaterRemover

    mesh = FloaterRemover()(mesh)
    mesh = DegenerateFaceRemover()(mesh)
    mesh = FaceReducer()(mesh, max_facenum=int(opts.get("max_faces", 40000)))
    if len(mesh.faces) == 0:
        raise RuntimeError("Shape model produced empty geometry - try a clearer sketch or a prompt")
    done("shape", t)
    place(shape, "cpu")
    del shape
    MODELS.release_if_sequential("shape")
    MODELS.flush()

    # 5. texture
    if opts.get("texture", True) and texture_available():
        t = stage("texture", "Painting texture with Hunyuan3D-Paint")
        try:
            mesh = MODELS.get_paint()(mesh, image=segmented)
            done("texture", t)
        except torch.cuda.OutOfMemoryError:
            MODELS.flush()
            job["stages_skipped"].append("texture")
            job["message"] = "Texture skipped: out of GPU memory"
        MODELS.release_if_sequential("paint")
        MODELS.flush()
    else:
        job["stages_skipped"].append("texture")

    # 6. validate + export
    t = stage("export", "Validating and exporting")
    glb_path = job_dir / ARTIFACT_FILES["glb"]
    mesh.export(glb_path)
    reloaded = trimesh.load(glb_path, force="mesh")
    if len(reloaded.faces) == 0:
        raise RuntimeError("Exported GLB has no faces")
    export_obj_zip(mesh, job_dir)
    job["artifacts"]["glb"] = f"/jobs/{job['id']}/files/glb"
    job["artifacts"]["obj_zip"] = f"/jobs/{job['id']}/files/obj_zip"
    job["stats"] = mesh_stats(mesh)
    done("export", t)
    job["total_seconds"] = round(time.time() - t_start, 2)


# --------------------------------------------------------------------------- job queue
JOBS: dict[str, dict] = {}
WORK: "queue.Queue[tuple[str, Image.Image, str, dict]]" = queue.Queue()


def worker() -> None:
    while True:
        job_id, img, prompt, opts = WORK.get()
        job = JOBS[job_id]
        job["status"] = "running"
        try:
            with MODELS.lock:
                run_job(job, img, prompt, opts)
            job["status"], job["message"] = "done", "Finished"
        except Exception as exc:
            traceback.print_exc()
            job["status"], job["error"] = "error", f"{type(exc).__name__}: {exc}"
            MODELS.flush()
        finally:
            (OUT_DIR / job_id / "job.json").write_text(json.dumps(job, indent=2))


threading.Thread(target=worker, daemon=True).start()

# --------------------------------------------------------------------------- API
app = FastAPI(title="Sketch2Shape AI backend", version=VERSION)
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])


@app.get("/health")
def health():
    return {
        "status": "ok",
        "gpu": GPU_NAME,
        "vram_gb": VRAM_GB,
        "low_vram": LOW_VRAM,
        "ram_gb": RAM_GB,
        "sequential": SEQUENTIAL,
        "shape_model": f"{SHAPE_MODEL}/{SHAPE_SUBFOLDER}",
        "texture_available": texture_available(),
        "controlnet_model": CN_MODEL,
        "loaded": MODELS.loaded(),
        "queue": WORK.qsize(),
        "version": VERSION,
    }


@app.post("/jobs")
async def create_job(image: UploadFile = File(...), prompt: str = Form(""), options: str = Form("{}")):
    try:
        opts = json.loads(options or "{}")
    except json.JSONDecodeError as exc:
        raise HTTPException(400, f"options must be JSON: {exc}")
    data = await image.read()
    if len(data) > 15 * 1024 * 1024:
        raise HTTPException(413, "Image larger than 15 MB")
    try:
        from io import BytesIO

        img = Image.open(BytesIO(data))
        img.load()
    except Exception:
        raise HTTPException(400, "Could not decode image")
    if img.mode in ("RGBA", "LA"):  # flatten transparency onto white paper
        bg = Image.new("RGB", img.size, "white")
        bg.paste(img, mask=img.getchannel("A"))
        img = bg
    job_id = uuid.uuid4().hex[:12]
    (OUT_DIR / job_id).mkdir(parents=True, exist_ok=True)
    JOBS[job_id] = {
        "id": job_id,
        "status": "queued",
        "stage": None,
        "message": f"Queued (position {WORK.qsize() + 1})",
        "error": None,
        "timings": {},
        "stages_skipped": [],
        "artifacts": {},
        "stats": None,
        "total_seconds": None,
        "prompt": prompt,
        "options": opts,
    }
    WORK.put((job_id, img, prompt, opts))
    return {"job_id": job_id}


@app.get("/jobs/{job_id}")
def get_job(job_id: str):
    job = JOBS.get(job_id)
    if not job:
        raise HTTPException(404, "Unknown job")
    return job


@app.get("/jobs/{job_id}/files/{name}")
def get_file(job_id: str, name: str):
    if job_id not in JOBS or name not in ARTIFACT_FILES:
        raise HTTPException(404, "Not found")
    path = OUT_DIR / job_id / ARTIFACT_FILES[name]
    if not path.exists():
        raise HTTPException(404, "File not ready")
    return FileResponse(path, filename=path.name)


# --------------------------------------------------------------------------- helpers
def warmup(texture: bool = True) -> None:
    """Download + load every model once so the first judge request is fast.

    In sequential mode each model is loaded and released in turn, so weights end
    up in the disk cache without ever holding more than one model in RAM.
    """
    t = time.time()
    print(f"[s2s] RAM {RAM_GB or 0:.1f} GB | VRAM {VRAM_GB or 0:.1f} GB | sequential={SEQUENTIAL}")
    steps = [("sd", MODELS.get_sd), ("rembg", MODELS.get_rembg), ("shape", MODELS.get_shape)]
    if texture and texture_available():
        steps.append(("paint", MODELS.get_paint))
    for name, load in steps:
        t0 = time.time()
        model = load()
        if name == "shape":
            place(model, "cpu")
        del model
        print(f"[s2s] {name} ready in {time.time() - t0:.0f}s | RAM used {_ram_used_gb() or 0:.1f} GB")
        if name != "rembg":
            MODELS.release_if_sequential(name)
    MODELS.flush()
    print(f"[s2s] models ready in {time.time() - t:.0f}s  loaded={MODELS.loaded()}")


def start_in_background(port: int = PORT) -> threading.Thread:
    import uvicorn

    server = uvicorn.Server(uvicorn.Config(app, host="0.0.0.0", port=port, log_level="warning"))
    th = threading.Thread(target=server.run, daemon=True)
    th.start()
    return th


if __name__ == "__main__":
    import uvicorn

    uvicorn.run(app, host="0.0.0.0", port=PORT)

## 5 · Choose models and warm up
Downloads every model once so the first request from the judges is fast.

**Free Colab (12.7 GB RAM):** the server auto-enables *sequential mode*. Each model is loaded straight onto the GPU,
used for its stage, then freed. This keeps peak RAM to one model at a time. Each request reloads models from the disk
cache, which adds roughly 20–60 s. With Colab Pro High-RAM (more than 20 GB), models stay resident and requests are faster.

If an earlier attempt crashed, use `Runtime → Restart session` first, then run from step 4.
On an L4 or A100, switch to the full `Hunyuan3D-2` turbo model for better geometry.

In [ ]:
# ===== Step 5 · Models + RAM-limited warm-up (keeps this process under 10 GB) =====
import os, sys, gc, time, ctypes, threading
import psutil, torch

RAM_LIMIT_GB = 10.0      # hard cap: warm-up aborts cleanly instead of crashing Colab
RAM_SOFT_GB = 8.5        # above this: force GC and hand freed memory back to the OS
LOAD_TEST_PAINT = True   # False = only download Hunyuan3D-Paint (lowest RAM warm-up)

os.environ.update({
    "S2S_SHAPE_MODEL": "tencent/Hunyuan3D-2mini",          # L4/A100: "tencent/Hunyuan3D-2"
    "S2S_SHAPE_SUBFOLDER": "hunyuan3d-dit-v2-mini-turbo",  # L4/A100: "hunyuan3d-dit-v2-0-turbo"
    "S2S_TEX_MODEL": "tencent/Hunyuan3D-2",
    "S2S_SD_MODEL": "Lykon/dreamshaper-8",
    "S2S_CN_MODEL": "lllyasviel/control_v11p_sd15_scribble",
    "S2S_ENABLE_TEXTURE": "1",
    "S2S_SEQUENTIAL": "1",   # REQUIRED on 12 GB RAM: only one model in memory at a time
})
os.chdir("/content/Hunyuan3D-2")
for p in ("/content", "/content/Hunyuan3D-2"):
    if p not in sys.path:
        sys.path.insert(0, p)

# ---------- RAM helpers ----------
_libc = ctypes.CDLL("libc.so.6")
_proc = psutil.Process()

def rss_gb():
    return _proc.memory_info().rss / 1024**3

def sys_gb():
    return psutil.virtual_memory().used / 1024**3

def trim():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    _libc.malloc_trim(0)  # glibc keeps freed blocks; this returns them to the OS

class RamLimiter:
    """Background watchdog: trims at the soft limit, raises MemoryError in the
    main thread at the hard limit so loading stops before the kernel is killed."""

    def __init__(self, hard_gb, soft_gb, interval=0.25):
        self.hard, self.soft, self.interval = hard_gb, soft_gb, interval
        self.peak, self.tripped = 0.0, False
        self._stop = threading.Event()
        self._main = threading.main_thread().ident

    def _run(self):
        while not self._stop.is_set():
            r = rss_gb()
            self.peak = max(self.peak, r)
            if r > self.hard and not self.tripped:
                self.tripped = True
                ctypes.pythonapi.PyThreadState_SetAsyncExc(
                    ctypes.c_ulong(self._main), ctypes.py_object(MemoryError))
            elif r > self.soft:
                trim()
            time.sleep(self.interval)

    def __enter__(self):
        threading.Thread(target=self._run, daemon=True).start()
        return self

    def __exit__(self, *exc):
        self._stop.set()
        return False

print(f"Start: process {rss_gb():.1f} GB | system {sys_gb():.1f} / "
      f"{psutil.virtual_memory().total / 1024**3:.1f} GB | limit {RAM_LIMIT_GB} GB")

# ---------- Phase 1: download weights to disk (no RAM needed) ----------
from huggingface_hub import snapshot_download
from diffusers import DiffusionPipeline

E = os.environ
downloads = [
    ("SD 1.5 (DreamShaper-8)", lambda: DiffusionPipeline.download(E["S2S_SD_MODEL"], use_safetensors=True)),
    ("ControlNet Scribble", lambda: snapshot_download(
        E["S2S_CN_MODEL"], allow_patterns=["config.json", "diffusion_pytorch_model.safetensors"])),
    ("Hunyuan3D shape", lambda: snapshot_download(
        E["S2S_SHAPE_MODEL"], allow_patterns=[f"{E['S2S_SHAPE_SUBFOLDER']}/*"])),
    ("Hunyuan3D paint + delight", lambda: snapshot_download(
        E["S2S_TEX_MODEL"], allow_patterns=["hunyuan3d-paint-v2-0-turbo/*", "hunyuan3d-delight-v2-0/*"])),
]
for name, fn in downloads:
    t0 = time.time()
    fn()
    trim()
    print(f"  downloaded {name:28s} {time.time() - t0:6.0f}s | process {rss_gb():.1f} GB")

# ---------- Phase 2: load-test each model one at a time under the limiter ----------
import importlib
if "s2s_server" in sys.modules:  # step 4 rewrote the file: pick up the new code
    importlib.reload(sys.modules["s2s_server"])
import s2s_server as S
S.SEQUENTIAL = True  # enforce even if the module was imported earlier with "0"

tests = [("sd", S.MODELS.get_sd), ("rembg", S.MODELS.get_rembg), ("shape", S.MODELS.get_shape)]
if LOAD_TEST_PAINT and S.texture_available():
    tests.append(("paint", S.MODELS.get_paint))

results = {}
with RamLimiter(RAM_LIMIT_GB, RAM_SOFT_GB) as lim:
    for name, load in tests:
        t0 = time.time()
        try:
            model = load()
            del model
            results[name] = "OK"
            print(f"  OK   {name:6s} loaded in {time.time() - t0:5.0f}s | peak {lim.peak:.1f} GB")
        except MemoryError:
            results[name] = "over limit"
            lim.tripped = False
            print(f"  FAIL {name:6s} exceeded {RAM_LIMIT_GB} GB - aborted cleanly")
        except Exception as exc:  # report and continue with the next model
            msg = str(exc)
            if "MemoryError" in msg:  # watchdog fired inside a wrapped call
                results[name] = "over limit"
                lim.tripped = False
                print(f"  FAIL {name:6s} exceeded {RAM_LIMIT_GB} GB - aborted cleanly")
            else:
                results[name] = f"error: {type(exc).__name__}"
                short = msg if len(msg) < 500 else msg[:250] + " ... " + msg[-250:]
                print(f"  FAIL {name:6s} {type(exc).__name__}: {short}")
        finally:
            if name != "rembg":          # rembg is tiny, keep it loaded
                S.MODELS.release(name)
            trim()
            print(f"       after release: process {rss_gb():.1f} GB | system {sys_gb():.1f} GB")

    print(f"\nWarm-up done | peak process RAM {lim.peak:.1f} GB (limit {RAM_LIMIT_GB} GB) | {results}")

if results.get("sd") != "OK" or results.get("shape") != "OK":
    raise RuntimeError("Core models did not fit - restart the session and re-run steps 4-5")
if results.get("paint", "OK") != "OK":
    print("Hunyuan3D-Paint failed to load: turn OFF 'Paint texture' in the app (shapes still work).")

## 6 · Start the API and open a public tunnel
Uses a Cloudflare quick tunnel (no account needed). Copy the printed URL into the web app under **Connect Colab GPU**.

In [ ]:
import re, subprocess, sys, threading, time
for p in ("/content", "/content/Hunyuan3D-2"):
    if p not in sys.path:
        sys.path.insert(0, p)
import s2s_server  # same module instance step 5 warmed up
s2s_server.start_in_background(8000)

!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared
!chmod +x /content/cloudflared

tunnel = subprocess.Popen(
    ["/content/cloudflared", "tunnel", "--url", "http://localhost:8000", "--no-autoupdate"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
PUBLIC_URL, t0 = None, time.time()
while time.time() - t0 < 90 and PUBLIC_URL is None:
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", tunnel.stdout.readline())
    if m:
        PUBLIC_URL = m.group(0)
# keep draining logs so the tunnel never blocks on a full pipe
threading.Thread(target=lambda: [None for _ in tunnel.stdout], daemon=True).start()
print("\n=========================================")
print(" Paste this into Sketch2Shape:", PUBLIC_URL)
print("=========================================")

## 7 · Smoke test and benchmark
Runs a synthetic sketch through the full pipeline and prints per-stage timings for your README.

In [ ]:
import io, json, requests
from PIL import Image, ImageDraw

img = Image.new("RGB", (512, 512), "white")
d = ImageDraw.Draw(img)
d.rounded_rectangle([170, 150, 340, 400], radius=20, outline="black", width=6)  # mug body
d.arc([300, 210, 400, 330], start=-90, end=90, fill="black", width=6)            # handle
d.ellipse([170, 130, 340, 170], outline="black", width=6)                        # rim
buf = io.BytesIO(); img.save(buf, format="PNG"); buf.seek(0)

base = "http://localhost:8000"
print(requests.get(f"{base}/health").json())
job = requests.post(f"{base}/jobs", files={"image": ("sketch.png", buf, "image/png")},
                    data={"prompt": "a glossy blue ceramic coffee mug", "options": json.dumps({"texture": True})}).json()
while True:
    s = requests.get(f"{base}/jobs/{job['job_id']}").json()
    print(f"\r{s['status']:8s} {s['stage'] or '':10s} {s['message'][:60]:60s}", end="")
    if s["status"] in ("done", "error"):
        break
    time.sleep(2)
print("\n", json.dumps({k: s[k] for k in ("status", "error", "timings", "total_seconds", "stats")}, indent=2))

## 8 · (Optional) Download all generated assets

In [ ]:
!cd /content && zip -qr s2s_outputs.zip s2s_outputs
from google.colab import files
files.download("/content/s2s_outputs.zip")

## Troubleshooting
- **Session crashes / "used all available RAM"**: make sure the warm-up log shows `sequential=True`. Restart the session before re-running, and do not run step 7 while another cell is loading models.
- **`numpy.core.multiarray failed to import` / OpenCV ABI errors**: numpy was downgraded. Run `!pip install -q --upgrade "numpy>=2.3,<2.6" "numba>=0.62"` and restart the session.
- **`cannot import name '_Ink' from 'PIL._typing'`** (Pillow files from mixed versions): `!pip uninstall -y pillow`, `!rm -rf /usr/local/lib/python3.13/dist-packages/PIL`, `!pip install --no-cache-dir "pillow==11.3.0"`, then restart the session.
- **`_multiarray_umath has no attribute '_blas_supports_fpe'`** or other numpy internals errors (numpy files from mixed versions): `!pip uninstall -y numpy` (twice), `!rm -rf /usr/local/lib/python3.13/dist-packages/numpy*`, `!pip install --no-cache-dir "numpy>=2.3,<2.6" "numba>=0.62"`, then restart the session. If more packages break, use `Runtime → Disconnect and delete runtime` and run the notebook from the top with no extra pins.
- **`Missing key(s) in state_dict ... main_image_encoder`**: transformers 5.x is installed. `!pip install -q "transformers>=4.46,<5"`, then restart the session.
- **`contains custom code ... Pass trust_remote_code=True`**: re-run step 4. The server patches Hunyuan's paint loader automatically.
- **Always restart the session after any `pip install` that changes versions**. Otherwise old modules stay loaded in memory.
- **CUDA out of memory on T4**: turn off *Paint texture* in the app, lower *Octree* to 196, or switch to an L4 runtime.
- **Tunnel URL changed**: quick tunnels are temporary. Re-run step 6 and paste the new URL.
- **`custom_rasterizer` import error after `setup.py install`**: `Runtime → Restart session`, then re-run from step 4 (installs persist).
- **Colab disconnects when idle**: keep the tab open during judging; models must reload after a disconnect (re-run steps 4–6).